In [ ]:
# Cell 1 — Load config
%run /home/jovyan/work/setup/config.py

In [ ]:
# Cell 2 — Load DQ log; keep only the latest result per (layer, table, check)
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

df_dq = spark.read.format("delta").load(f"{GOLD_PATH}/dq_log")

# De-duplicate: take the most-recent row per unique check key
w = Window.partitionBy("layer", "table_name", "check_name").orderBy(col("run_ts").desc())
df_latest = df_dq.withColumn("_rn", row_number().over(w)).filter("_rn = 1").drop("_rn")

total  = df_latest.count()
passed = df_latest.filter("status = 'PASS'").count()
failed = df_latest.filter("status = 'FAIL'").count()

print(f"Total DQ checks (latest run): {total} | PASS: {passed} | FAIL: {failed}")
print("\n=== ALL LATEST DQ RESULTS ===")
df_latest.orderBy("layer", "table_name", "check_name").show(50, truncate=False)

In [ ]:
# Cell 3 — Show failures only (if any)
df_fail = df_latest.filter("status = 'FAIL'")
if df_fail.count() == 0:
    print("All DQ checks PASSED — pipeline is healthy.")
else:
    print(f"WARNING: {df_fail.count()} DQ check(s) FAILED")
    df_fail.show(truncate=False)
    raise Exception(f"DQ FAILED: {df_fail.count()} check(s) failed. See output above.")

In [ ]:
# Cell 4 — Row count summary across all Gold tables
gold_tables = [
    "dim_date", "dim_product", "dim_channel", "dim_geography",
    "fact_sales",
    "summary_sales_by_brand_month",
    "summary_sales_by_region_trade_group",
    "summary_sales_by_brand_region",
    "summary_top3_trade_group_per_region",
    "summary_lowest_brand_per_region",
    "summary_sales_by_channel_type",
    "summary_sales_by_package_category",
]
print("\n=== GOLD TABLE ROW COUNTS ===")
for tbl in gold_tables:
    count = spark.read.format("delta").load(f"{GOLD_PATH}/{tbl}").count()
    print(f"  {tbl}: {count} rows")

In [ ]:
# Cell 5 — Verify answers to 3 business questions are present
print("\n=== BUSINESS QUERY VALIDATION ===")

df_q1 = spark.read.format("delta").load(f"{GOLD_PATH}/summary_top3_trade_group_per_region")
print(f"Req 4.1 (Top 3 per region): {df_q1.count()} rows (expect 7 regions x up to 3 = ~21)")

df_q2 = spark.read.format("delta").load(f"{GOLD_PATH}/summary_sales_by_brand_month")
print(f"Req 4.2 (Brand x month):    {df_q2.count()} rows (expect 12 months x 4 brands = ~48)")

df_q3 = spark.read.format("delta").load(f"{GOLD_PATH}/summary_lowest_brand_per_region")
print(f"Req 4.3 (Lowest brand):     {df_q3.count()} rows (expect 1 per region = 7)")

print("\nPipeline complete. All requirements covered.")